# Cross-Industry Accelerator — Create Agents
### Auto-Create Fabric IQ QA Agent (DataAgent) and Ops Agent (OperationsAgent)

This notebook creates **two agents** of **different Fabric item types**:

| Agent | Fabric Item Type | API Endpoint | DataSources |
|---|---|---|---|
| **QA Agent** | `DataAgent` | `POST /items` (SDK) | Lakehouse, Warehouse, KQL Database, Semantic Model |
| **Ops Agent** | `OperationsAgent` | `POST /operationsAgents` | KQL Database only (KustoDatabase) |

**What it does:**
1. Reads industry configuration from `00_Industry_Config`
2. Resolves workspace items: Ontology, Lakehouse, Eventhouse, KQL Database
3. Creates the **QA Agent** (DataAgent) via `fabric-data-agent-sdk`
4. Creates the **Ops Agent** (OperationsAgent) via dedicated REST API with OperationsAgentV1 definition
5. Configures each agent with industry-appropriate instructions, goals, and datasources

> **Prerequisites:**
> 1. Run `05_Create_Ontology.ipynb` to create the ontology first
> 2. Run `04_Create_Semantic_Model.ipynb` to create the semantic model
> 3. Data must be loaded (Lakehouse + Eventhouse) for agents to query
> 4. Each industry must have a `{Industry}_Agent_Instructions.ipynb` notebook
>
> **API References:**
> - [OperationsAgent REST API](https://learn.microsoft.com/en-us/rest/api/fabric/operationsagent/items)
> - [OperationsAgent Definition](https://learn.microsoft.com/en-us/rest/api/fabric/articles/item-management/definitions/operations-agent-definition)

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

In [ ]:
%run 00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# LOAD INDUSTRY-SPECIFIC AGENT INSTRUCTIONS
# ════════════════════════════════════════════════════════════════════════
# Runs the industry instructions notebook, which writes a JSON file to
# the Lakehouse. We then read it back (avoids notebook.exit() size limits).

import json

_instructions_nb = f"{INDUSTRY_LABEL}_Agent_Instructions"
_tmp_path = "file:/lakehouse/default/Files/_temp/agent_instructions.json"
print(f"Loading agent instructions from: {_instructions_nb}")

try:
    notebookutils.notebook.run(_instructions_nb, 600, {'useRootDefaultLakehouse': True})
    _content = notebookutils.fs.head(_tmp_path, 1000000)
    _parsed = json.loads(_content)
    QA_AGENT_INSTRUCTIONS = _parsed["qa"]
    OPS_AGENT_INSTRUCTIONS = _parsed["ops"]
    QA_AGENT_GOALS = _parsed.get("qa_goals")
    OPS_AGENT_GOALS = _parsed.get("ops_goals")
    QA_FEWSHOTS = _parsed.get("qa_fewshots", {})
    OPS_FEWSHOTS = _parsed.get("ops_fewshots", {})
    QA_DS_INSTRUCTIONS = _parsed.get("qa_ds_instructions", {})
    OPS_DS_INSTRUCTIONS = _parsed.get("ops_ds_instructions", {})
    print(f"  QA instructions:  {len(QA_AGENT_INSTRUCTIONS):,} chars")
    print(f"  Ops instructions: {len(OPS_AGENT_INSTRUCTIONS):,} chars")
    print(f"  QA goals:  {len(QA_AGENT_GOALS or ''):,} chars")
    print(f"  Ops goals: {len(OPS_AGENT_GOALS or ''):,} chars")
    print(f"  QA fewshots:  {sum(len(v) for v in QA_FEWSHOTS.values())} queries across {len(QA_FEWSHOTS)} datasources")
    print(f"  Ops fewshots: {sum(len(v) for v in OPS_FEWSHOTS.values())} queries across {len(OPS_FEWSHOTS)} datasources")
    print(f"  DS instructions: QA={len(QA_DS_INSTRUCTIONS)}, Ops={len(OPS_DS_INSTRUCTIONS)}")
except Exception as e:
    print(f"  Could not load {_instructions_nb}: {e}")
    print(f"  Will use generic instructions.")
    QA_AGENT_INSTRUCTIONS = None
    OPS_AGENT_INSTRUCTIONS = None
    QA_AGENT_GOALS = None
    OPS_AGENT_GOALS = None
    QA_FEWSHOTS = {}
    OPS_FEWSHOTS = {}
    QA_DS_INSTRUCTIONS = {}
    OPS_DS_INSTRUCTIONS = {}

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RESOLVE WORKSPACE ITEMS
# ════════════════════════════════════════════════════════════════════════

import sempy.fabric as fabric
import json, requests

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')
items_df = fabric.list_items()

# Resolve Ontology item ID
ont_matches = items_df[(items_df["Type"] == "Ontology") & (items_df["Display Name"] == ONTOLOGY_NAME)]
if ont_matches.empty:
    print(f"⚠ Ontology '{ONTOLOGY_NAME}' not found. Run 05_Create_Ontology first.")
    print(f"Available ontologies:")
    print(items_df[items_df["Type"] == "Ontology"][["Display Name", "Id"]].to_string(index=False))
    ontology_item_id = None
else:
    ontology_item_id = str(ont_matches.iloc[0].Id)
    print(f"✓ Ontology: {ONTOLOGY_NAME} → {ontology_item_id}")

# Resolve Lakehouse item ID
lh_matches = items_df[(items_df["Type"] == "Lakehouse") & (items_df["Display Name"] == LAKEHOUSE_NAME)]
lakehouse_item_id = str(lh_matches.iloc[0].Id) if not lh_matches.empty else None
print(f"✓ Lakehouse: {LAKEHOUSE_NAME} → {lakehouse_item_id}")

# Resolve Eventhouse item ID
eventhouse_item_id = None
if EVENTHOUSE_NAME:
    eh_matches = items_df[(items_df["Type"] == "Eventhouse") & (items_df["Display Name"] == EVENTHOUSE_NAME)]
    if not eh_matches.empty:
        eventhouse_item_id = str(eh_matches.iloc[0].Id)
        print(f"✓ Eventhouse: {EVENTHOUSE_NAME} → {eventhouse_item_id}")

# Resolve KQL Database item ID (needed for OperationsAgent datasource)
kql_database_item_id = None
if EVENTHOUSE_DATABASE:
    kql_matches = items_df[(items_df["Type"] == "KQLDatabase") & (items_df["Display Name"] == EVENTHOUSE_DATABASE)]
    if not kql_matches.empty:
        kql_database_item_id = str(kql_matches.iloc[0].Id)
        print(f"✓ KQL Database: {EVENTHOUSE_DATABASE} → {kql_database_item_id}")
    else:
        print(f"⚠ KQL Database '{EVENTHOUSE_DATABASE}' not found. OperationsAgent will be created without datasource.")

print(f"\n  QA Agent name:  {DATA_AGENT_NAME} (type: DataAgent)")
print(f"  Ops Agent name: {OPS_AGENT_NAME} (type: OperationsAgent)")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# INSTALL FABRIC DATA AGENT SDK (requires kernel restart after install)
# ════════════════════════════════════════════════════════════════════════
# ⚡ RUN THIS CELL ONCE, then:
#    1. Restart the kernel (Session → Restart session)
#    2. SKIP this cell (do NOT re-run it)
#    3. Run ALL remaining cells from the next cell onward
#       (Select next cell → Shift+Enter through each, or Run All Below)
#
# The next cell (Import SDK) re-loads config, workspace items, and
# instructions automatically — no need to re-run earlier cells.
# ════════════════════════════════════════════════════════════════════════

import sys, os

_AGENT_LIB = "/tmp/agent_sdk_lib"
os.makedirs(_AGENT_LIB, exist_ok=True)

# Install to isolated path using os.system (subprocess causes Jackson errors in Fabric)
_pip = sys.executable + " -m pip"
_cmd = f"{_pip} install --target={_AGENT_LIB} --upgrade --no-cache-dir -q -q fabric-data-agent-sdk 'typing_extensions>=4.13' 'requests>=2.32.3' 2>/dev/null"
rc = os.system(_cmd)
assert rc == 0, f"pip install failed with exit code {rc}"

# Verify
_te_path = os.path.join(_AGENT_LIB, "typing_extensions.py")
with open(_te_path) as f:
    assert "Sentinel" in f.read(), "Sentinel not found in installed typing_extensions!"
print(f"  ✓ typing_extensions at {_te_path} has Sentinel")

print("\n" + "═" * 60)
print("  ✓ SDK INSTALLED SUCCESSFULLY")
print("═" * 60)
print()
print("  NEXT STEPS:")
print("  1. Restart the kernel  (Session → Restart session)")
print("  2. SKIP this cell      (do NOT re-run it)")
print("  3. Run all cells below (the next 4 cells reload everything)")
print("═" * 60)

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 1/3: Prepend SDK path + reload config (after kernel restart)
# ════════════════════════════════════════════════════════════════════════
import sys
_AGENT_LIB = "/tmp/agent_sdk_lib"
if _AGENT_LIB not in sys.path:
    sys.path.insert(0, _AGENT_LIB)
import typing_extensions as _te
print(f"  typing_extensions: {_te.__file__} (Sentinel={hasattr(_te, 'Sentinel')})")

In [ ]:
%run 00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 2/3: Load instructions + resolve workspace items
# ════════════════════════════════════════════════════════════════════════
import json

print("── Loading agent instructions...")
_instructions_nb = f"{INDUSTRY_LABEL}_Agent_Instructions"
_tmp_path = "file:/lakehouse/default/Files/_temp/agent_instructions.json"
try:
    notebookutils.notebook.run(_instructions_nb, 600, {'useRootDefaultLakehouse': True})
    _content = notebookutils.fs.head(_tmp_path, 1000000)
    _parsed = json.loads(_content)
    QA_AGENT_INSTRUCTIONS = _parsed["qa"]
    OPS_AGENT_INSTRUCTIONS = _parsed["ops"]
    QA_AGENT_GOALS = _parsed.get("qa_goals")
    OPS_AGENT_GOALS = _parsed.get("ops_goals")
    QA_FEWSHOTS = _parsed.get("qa_fewshots", {})
    OPS_FEWSHOTS = _parsed.get("ops_fewshots", {})
    QA_DS_INSTRUCTIONS = _parsed.get("qa_ds_instructions", {})
    OPS_DS_INSTRUCTIONS = _parsed.get("ops_ds_instructions", {})
    print(f"  QA instructions:  {len(QA_AGENT_INSTRUCTIONS):,} chars")
    print(f"  Ops instructions: {len(OPS_AGENT_INSTRUCTIONS):,} chars")
    print(f"  QA goals:  {len(QA_AGENT_GOALS or ''):,} chars")
    print(f"  Ops goals: {len(OPS_AGENT_GOALS or ''):,} chars")
    print(f"  Fewshots: QA={sum(len(v) for v in QA_FEWSHOTS.values())}, Ops={sum(len(v) for v in OPS_FEWSHOTS.values())} queries")
    print(f"  DS instructions: QA={len(QA_DS_INSTRUCTIONS)}, Ops={len(OPS_DS_INSTRUCTIONS)} datasources")
except Exception as e:
    print(f"  Could not load {_instructions_nb}: {e}")
    QA_AGENT_INSTRUCTIONS = OPS_AGENT_INSTRUCTIONS = QA_AGENT_GOALS = OPS_AGENT_GOALS = None
    QA_FEWSHOTS = OPS_FEWSHOTS = QA_DS_INSTRUCTIONS = OPS_DS_INSTRUCTIONS = {}

print("\n── Resolving workspace items...")
import sempy.fabric as fabric
import requests

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')
items_df = fabric.list_items()

ont_matches = items_df[(items_df["Type"] == "Ontology") & (items_df["Display Name"] == ONTOLOGY_NAME)]
ontology_item_id = str(ont_matches.iloc[0].Id) if not ont_matches.empty else None
print(f"  Ontology: {ONTOLOGY_NAME} → {ontology_item_id}")

lh_matches = items_df[(items_df["Type"] == "Lakehouse") & (items_df["Display Name"] == LAKEHOUSE_NAME)]
lakehouse_item_id = str(lh_matches.iloc[0].Id) if not lh_matches.empty else None
print(f"  Lakehouse: {LAKEHOUSE_NAME} → {lakehouse_item_id}")

eventhouse_item_id = None
if EVENTHOUSE_NAME:
    eh_matches = items_df[(items_df["Type"] == "Eventhouse") & (items_df["Display Name"] == EVENTHOUSE_NAME)]
    eventhouse_item_id = str(eh_matches.iloc[0].Id) if not eh_matches.empty else None
    print(f"  Eventhouse: {EVENTHOUSE_NAME} → {eventhouse_item_id}")

# Resolve KQL Database item ID (needed for OperationsAgent datasource)
kql_database_item_id = None
if EVENTHOUSE_DATABASE:
    kql_matches = items_df[(items_df["Type"] == "KQLDatabase") & (items_df["Display Name"] == EVENTHOUSE_DATABASE)]
    kql_database_item_id = str(kql_matches.iloc[0].Id) if not kql_matches.empty else None
    print(f"  KQL Database: {EVENTHOUSE_DATABASE} → {kql_database_item_id}")

print(f"\n  QA Agent name:  {DATA_AGENT_NAME} (type: DataAgent)")
print(f"  Ops Agent name: {OPS_AGENT_NAME} (type: OperationsAgent)")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 3/3: Import SDK
# ════════════════════════════════════════════════════════════════════════
# Force reload typing_extensions from our custom path before importing SDK
import sys
if 'typing_extensions' in sys.modules:
    del sys.modules['typing_extensions']

from fabric.dataagent.client import (
    FabricDataAgentManagement,
    create_data_agent,
    delete_data_agent,
)
print("✓ fabric-data-agent-sdk imported. Ready to create agents.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE QA AGENT + ADD DATA SOURCES + DS INSTRUCTIONS + FEWSHOTS
# ════════════════════════════════════════════════════════════════════════

import time, requests as req_lib
import sempy.fabric as fabric

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')
api_headers = {"Authorization": f"Bearer {access_token}"}
BASE = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

def _create_agent_with_retry(name, max_retries=5, wait_sec=25):
    for attempt in range(1, max_retries + 1):
        try:
            return create_data_agent(name)
        except Exception as e:
            err = str(e).lower()
            if "not available yet" in err or "notavailableyet" in err:
                print(f"    Retrying in {wait_sec}s ({attempt}/{max_retries})...")
                time.sleep(wait_sec)
            elif "already in use" in err:
                return FabricDataAgentManagement(name)
            else:
                raise
    raise RuntimeError(f"Name '{name}' not available after {max_retries} retries.")

def _configure_datasource(ds_obj, ds_name, ds_instructions, ds_fewshots):
    """Set instructions and fewshots on a Datasource object."""
    # Per-DS instructions
    if ds_name in ds_instructions and ds_instructions[ds_name]:
        try:
            ds_obj.update_configuration(instructions=ds_instructions[ds_name])
            print(f"      ✓ DS instructions ({len(ds_instructions[ds_name])} chars)")
        except Exception as e:
            print(f"      ✗ DS instructions: {e}")

    # Fewshots: {"question text": "query text"}
    if ds_name in ds_fewshots and ds_fewshots[ds_name]:
        added = 0
        for ex in ds_fewshots[ds_name]:
            try:
                if isinstance(ex, dict) and "question" in ex and "query" in ex:
                    ds_obj.add_fewshots({ex["question"]: ex["query"]})
                else:
                    ds_obj.add_fewshots(ex)
                added += 1
            except Exception as e:
                print(f"      ✗ Fewshot: {e}")
                break
        print(f"      ✓ {added}/{len(ds_fewshots[ds_name])} fewshots")

# Prepare DS config dicts
_qa_ds = QA_DS_INSTRUCTIONS if 'QA_DS_INSTRUCTIONS' in dir() and QA_DS_INSTRUCTIONS else {}
_qa_fs = QA_FEWSHOTS if 'QA_FEWSHOTS' in dir() and QA_FEWSHOTS else {}

# ── Clean up existing agents
items_df = fabric.list_items()

# Delete QA Agent (DataAgent type) via SDK, fallback to REST
for agent_name in [DATA_AGENT_NAME]:
    deleted = False
    try:
        delete_data_agent(agent_name)
        print(f"  Deleted DataAgent '{agent_name}'")
        deleted = True
    except Exception:
        pass
    if not deleted:
        matches = items_df[items_df["Display Name"] == agent_name]
        if not matches.empty:
            for _, row in matches.iterrows():
                req_lib.delete(f"{BASE}/items/{row['Id']}", headers=api_headers, timeout=30)
                deleted = True
    if not deleted:
        print(f"  '{agent_name}' — clean slate.")

# Delete Ops Agent (OperationsAgent type) via dedicated REST API
ops_matches = items_df[(items_df["Display Name"] == OPS_AGENT_NAME)]
if not ops_matches.empty:
    for _, row in ops_matches.iterrows():
        item_type = str(row.get("Type", ""))
        item_id = str(row["Id"])
        if item_type == "OperationsAgent":
            r = req_lib.delete(f"{BASE}/operationsAgents/{item_id}", headers=api_headers, timeout=30)
            print(f"  Deleted OperationsAgent '{OPS_AGENT_NAME}' (status={r.status_code})")
        else:
            # Legacy DataAgent or other type — try SDK then REST fallback
            try:
                delete_data_agent(OPS_AGENT_NAME)
                print(f"  Deleted legacy DataAgent '{OPS_AGENT_NAME}'")
            except Exception:
                req_lib.delete(f"{BASE}/items/{item_id}", headers=api_headers, timeout=30)
                print(f"  Deleted '{OPS_AGENT_NAME}' via REST (type={item_type})")
else:
    print(f"  '{OPS_AGENT_NAME}' — clean slate.")

print("  Waiting 25s for name availability...")
time.sleep(25)

# ── Create QA Agent
print(f"\nCreating QA Agent: {DATA_AGENT_NAME}...")
qa_agent = _create_agent_with_retry(DATA_AGENT_NAME)
print(f"  ✓ Created")

# Set agent-level instructions
qa_instructions = QA_AGENT_INSTRUCTIONS if ('QA_AGENT_INSTRUCTIONS' in dir() and QA_AGENT_INSTRUCTIONS) else f"You are a data analyst agent for the {INDUSTRY} industry."
qa_agent.update_configuration(instructions=qa_instructions)
print(f"  Agent instructions: {len(qa_instructions)} chars")

# Add datasources and configure each immediately
for ds_name, ds_type in [
    (WAREHOUSE_NAME, "warehouse"),
    (LAKEHOUSE_NAME, "lakehouse"),
]:
    print(f"\n    Adding {ds_type}: {ds_name}...")
    try:
        ds_obj = qa_agent.add_datasource(ds_name, type=ds_type)
        print(f"      Added.")
        _configure_datasource(ds_obj, ds_name, _qa_ds, _qa_fs)
    except Exception as e:
        print(f"      {e}")


if EVENTHOUSE_DATABASE:
    print(f"\n    Adding kqldatabase: {EVENTHOUSE_DATABASE}...")
    try:
        ds_obj = qa_agent.add_datasource(EVENTHOUSE_DATABASE, type="kqldatabase")
        print(f"      Added.")
        _configure_datasource(ds_obj, EVENTHOUSE_DATABASE, _qa_ds, _qa_fs)
    except Exception as e:
        print(f"      {e}")

# Semantic model (no fewshots supported)
if 'SEMANTIC_MODEL_NAME' in dir() and SEMANTIC_MODEL_NAME:
    print(f"\n    Adding semanticmodel: {SEMANTIC_MODEL_NAME}...")
    try:
        qa_agent.add_datasource(SEMANTIC_MODEL_NAME, type="semanticmodel")
        print(f"      Added (no fewshots for semantic models).")
    except Exception as e:
        print(f"      {e}")

# Publish
print(f"\n  Publishing...")
qa_agent.publish()
print(f"  ✓ Published: {DATA_AGENT_NAME}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE OPS AGENT (OperationsAgent type via dedicated REST API)
# ════════════════════════════════════════════════════════════════════════
# OperationsAgent has its own endpoint: POST /operationsAgents
# Docs: https://learn.microsoft.com/en-us/rest/api/fabric/operationsagent/items
#
# Key differences from DataAgent:
#   - Dedicated endpoint: /operationsAgents (NOT /items)
#   - DataSources: only KustoDatabase type supported
#   - Has goals, instructions, actions (PowerAutomate), playbook, shouldRun
#   - Definition format: OperationsAgentV1 with Configurations.json
# ════════════════════════════════════════════════════════════════════════

import base64, json, time
import requests as req_lib
import sempy.fabric as fabric

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')
api_headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json",
}
BASE = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

def _poll_lro(response, headers, timeout_sec=120):
    """Poll a Long Running Operation until completion."""
    if response.status_code not in (200, 201):
        op_id = response.headers.get("x-ms-operation-id")
        if not op_id:
            return response
        poll_url = f"https://api.fabric.microsoft.com/v1/operations/{op_id}"
        deadline = time.time() + timeout_sec
        while time.time() < deadline:
            retry_after = int(response.headers.get("Retry-After", 10))
            time.sleep(retry_after)
            r = req_lib.get(poll_url, headers=headers, timeout=30)
            if r.ok:
                body = r.json()
                status = body.get("status", "")
                if status in ("Succeeded", "Failed", "Cancelled"):
                    if status != "Succeeded":
                        raise RuntimeError(f"LRO {status}: {body}")
                    # Get the result item
                    result_url = f"{poll_url}/result"
                    result_r = req_lib.get(result_url, headers=headers, timeout=30)
                    return result_r if result_r.ok else r
        raise TimeoutError(f"LRO timed out after {timeout_sec}s")
    return response

# ── Step 1a: Ensure no existing OperationsAgent with this name
print(f"\nChecking for existing OperationsAgent: {OPS_AGENT_NAME}...")
items_df = fabric.list_items()
ops_matches = items_df[(items_df["Display Name"] == OPS_AGENT_NAME)]
if not ops_matches.empty:
    for _, row in ops_matches.iterrows():
        item_id = str(row["Id"])
        item_type = str(row.get("Type", ""))
        print(f"  Found existing {item_type}: {item_id}. Deleting...")
        if item_type == "OperationsAgent":
            r = req_lib.delete(f"{BASE}/operationsAgents/{item_id}", headers=api_headers, timeout=30)
        else:
            r = req_lib.delete(f"{BASE}/items/{item_id}", headers=api_headers, timeout=30)
        print(f"    Delete status: {r.status_code}")
    print(f"  Waiting 30s for name to become available...")
    time.sleep(30)

# ── Step 1b: Create OperationsAgent item (with retry for name availability)
print(f"\nCreating OperationsAgent: {OPS_AGENT_NAME}...")
create_body = {
    "displayName": OPS_AGENT_NAME,
    "description": f"Operations monitoring agent for {INDUSTRY} — monitors real-time events, surfaces anomalies and SLA breaches."
}

# Retry logic for "ItemDisplayNameNotAvailableYet"
max_retries = 5
retry_delay = 10
for attempt in range(1, max_retries + 1):
    r = req_lib.post(f"{BASE}/operationsAgents", headers=api_headers, json=create_body, timeout=60)
    
    if r.status_code == 202:
        print(f"  Accepted (LRO). Polling for completion...")
        r = _poll_lro(r, api_headers)
    
    if r.status_code in (200, 201):
        ops_agent_item = r.json()
        ops_agent_id = ops_agent_item["id"]
        print(f"  ✓ Created OperationsAgent: {OPS_AGENT_NAME} → {ops_agent_id}")
        print(f"    Type: {ops_agent_item.get('type', 'OperationsAgent')}")
        break
    elif r.status_code == 400:
        try:
            err_body = r.json()
            if err_body.get("errorCode") == "ItemDisplayNameNotAvailableYet" and attempt < max_retries:
                print(f"  Name not available yet. Retrying in {retry_delay}s ({attempt}/{max_retries})...")
                time.sleep(retry_delay)
                retry_delay += 5  # Exponential backoff
                continue
        except:
            pass
        raise RuntimeError(f"Failed to create OperationsAgent: {r.status_code} {r.text}")
    else:
        raise RuntimeError(f"Failed to create OperationsAgent: {r.status_code} {r.text}")

# ── Step 2: Build OperationsAgentV1 definition
ops_instructions = OPS_AGENT_INSTRUCTIONS if ('OPS_AGENT_INSTRUCTIONS' in dir() and OPS_AGENT_INSTRUCTIONS) else f"You are an operations monitoring agent for the {INDUSTRY} industry. Monitor KQL event tables and surface anomalies, SLA breaches, and quality alerts."

# Use industry-specific goals if available, otherwise use generic fallback
if 'OPS_AGENT_GOALS' in dir() and OPS_AGENT_GOALS:
    ops_goals = OPS_AGENT_GOALS
else:
    ops_goals = f"Monitor {INDUSTRY} operations data in real-time. Detect anomalies, SLA violations, and quality issues. Surface actionable alerts for operations teams."

# Build datasources dict (OperationsAgent only supports KustoDatabase type)
ops_datasources = {}
if kql_database_item_id:
    ops_datasources[EVENTHOUSE_DATABASE] = {
        "id": kql_database_item_id,
        "type": "KustoDatabase",
        "workspaceId": workspace_id
    }
    print(f"  DataSource: {EVENTHOUSE_DATABASE} → {kql_database_item_id} (KustoDatabase)")
else:
    print(f"  ⚠ No KQL Database found — OperationsAgent will have no datasource.")
    print(f"    Add a KQL Database datasource manually in the Fabric portal.")

# Build actions dict (empty for now — can add Power Automate flows later)
ops_actions = {}

# Assemble the full OperationsAgentV1 definition
ops_definition = {
    "configuration": {
        "goals": ops_goals,
        "instructions": ops_instructions,
        "dataSources": ops_datasources,
        "actions": ops_actions
    },
    "playbook": {},
    "shouldRun": True
}

print(f"  Definition: goals={len(ops_goals)} chars, instructions={len(ops_instructions)} chars")
print(f"  DataSources: {len(ops_datasources)}, Actions: {len(ops_actions)}")
print(f"  shouldRun: True")

# ── Step 3: Update definition via REST API
encoded_payload = base64.b64encode(json.dumps(ops_definition).encode()).decode()
update_body = {
    "definition": {
        "format": "OperationsAgentV1",
        "parts": [
            {
                "path": "OperationsAgentV1.json",
                "payload": encoded_payload,
                "payloadType": "InlineBase64"
            }
        ]
    }
}

print(f"\n  Updating definition...")
r = req_lib.post(
    f"{BASE}/operationsAgents/{ops_agent_id}/updateDefinition",
    headers=api_headers, json=update_body, timeout=60
)

if r.status_code == 202:
    print(f"  Accepted (LRO). Polling...")
    r = _poll_lro(r, api_headers)

if r.status_code in (200, 202):
    print(f"  ✓ Definition updated successfully.")
else:
    print(f"  ⚠ Definition update returned {r.status_code}: {r.text}")
    print(f"    The OperationsAgent was created but may need manual configuration.")

print(f"\n  ✓ OperationsAgent ready: {OPS_AGENT_NAME}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# AGENT CREATION SUMMARY
# ════════════════════════════════════════════════════════════════════════

print(f"\n{'═'*60}")
print(f"AGENT SUMMARY — {INDUSTRY.upper()}")
print(f"{'═'*60}")
print(f"")
print(f"  Ontology:       {ONTOLOGY_NAME}")
print(f"  Lakehouse:      {LAKEHOUSE_NAME}")
print(f"  Eventhouse:     {EVENTHOUSE_NAME or 'N/A'}")
print(f"  KQL Database:   {EVENTHOUSE_DATABASE or 'N/A'}")
print(f"")
print(f"  QA Agent:       {DATA_AGENT_NAME}  (type: DataAgent)")
print(f"    → Answers ad-hoc data questions about {INDUSTRY} data")
print(f"    → Queries: Lakehouse + Warehouse + Eventhouse + Semantic Model")
print(f"")
print(f"  Ops Agent:      {OPS_AGENT_NAME}  (type: OperationsAgent)")
print(f"    → Monitors real-time events via KQL datasource")
print(f"    → Surfaces: anomalies, SLA breaches, quality alerts")
print(f"    → API: POST /operationsAgents (dedicated endpoint)")
print(f"")

print(f"✅ Agent creation complete.")
print(f"   Next: Run 07_Create_Dashboards.ipynb to create real-time and Power BI dashboards.")